# អ្នកចាត់ថ្នាក់រូបភាពសាមញ្ញ

សៀវភៅកំណត់ត្រានេះបង្ហាញអ្នកពីរបៀបចាត់ថ្នាក់រូបភាពដោយប្រើបណ្តាញសរសៃប្រសammatមុនដែលបានបណ្តុះបណ្តាលរួច។

**អ្វីដែលអ្នកនឹងរៀន:**
- របៀបផ្ទុក និងប្រើម៉ូដែលដែលបានបណ្តុះបណ្តាលរួច
- ការប្រមូលរូបភាពជាមុន
- ការធ្វើការព្យាករណ៍លើរូបភាព
- ការយល់ដឹងអំពីពិន្ទុជំនាញ

**ករណីប្រើប្រាស់:** កំណត់អត្តសញ្ញាណវត្ថុក្នុងរូបភាព (ដូចជា "ឆ្មា", "ឆ្កែ", "រថយន្ត", ល។)

---


## ជំហានទី 1៖ នាំចូលបណ្ណាល័យដែលត្រូវការ

មកនាំចូលឧបករណ៍ដែលយើងត្រូវការគ្នា។ កុំបារម្ភបើអ្នកមិនយល់ពីអ្វីៗទាំងអស់ទេនៅឡើយ!


In [ ]:
# Core libraries
import numpy as np
from PIL import Image
import requests
from io import BytesIO

# TensorFlow for deep learning
try:
    import tensorflow as tf
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
    print("✅ TensorFlow loaded successfully!")
    print(f"   Version: {tf.__version__}")
except ImportError:
    print("❌ Please install TensorFlow: pip install tensorflow")

## ជំហានទី 2: បញ្ចូលគំរូដែលបានបណ្តុះបណ្តាលរួចហើយ

យើងនឹងប្រើ **MobileNetV2**, បណ្តាញសរសៃប្រសាទដែលបានបណ្តុះបណ្តាលរួចហើយលើរូបភាពរាប់លានលាន។

នេះហៅថា **ការរៀនផ្ទេរ** - ប្រើគំរូដែលនរណាម្នាក់បានបណ្តុះបណ្តាលរួចហើយ!


In [ ]:
print("📦 Loading pre-trained MobileNetV2 model...")
print("   This may take a minute on first run (downloading weights)...")

# Load the model
# include_top=True means we use the classification layer
# weights='imagenet' means it was trained on ImageNet dataset
model = MobileNetV2(weights='imagenet', include_top=True)

print("✅ Model loaded!")
print(f"   The model can recognize 1000 different object categories")

## ជំហានទី 3: មុខងារជំនួយ

មកបង្កើតមុខងារដើម្បីផ្ទុកនិងរៀបចំរូបភាពសម្រាប់ម៉ូឌែលរបស់យើង។


In [ ]:
def load_image_from_url(url):
    """
    Load an image from a URL.
    
    Args:
        url: Web address of the image
        
    Returns:
        PIL Image object
    """
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img


def prepare_image(img):
    """
    Prepare an image for the model.
    
    Steps:
    1. Resize to 224x224 (model's expected size)
    2. Convert to array
    3. Add batch dimension
    4. Preprocess for MobileNetV2
    
    Args:
        img: PIL Image
        
    Returns:
        Preprocessed image array
    """
    # Resize to 224x224 pixels
    img = img.resize((224, 224))
    
    # Convert to numpy array
    img_array = np.array(img)
    
    # Add batch dimension (model expects multiple images)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Preprocess for MobileNetV2
    img_array = preprocess_input(img_array)
    
    return img_array


def classify_image(img):
    """
    Classify an image and return top predictions.
    
    Args:
        img: PIL Image
        
    Returns:
        List of (class_name, confidence) tuples
    """
    # Prepare the image
    img_array = prepare_image(img)
    
    # Make prediction
    predictions = model.predict(img_array, verbose=0)
    
    # Decode predictions to human-readable labels
    # top=5 means we get the top 5 most likely classes
    decoded = decode_predictions(predictions, top=5)[0]
    
    # Convert to simpler format
    results = [(label, float(confidence)) for (_, label, confidence) in decoded]
    
    return results


print("✅ Helper functions ready!")

## ជំហានទី 4៖ សាកល្បងលើរូបភាពគំរូ

មកព្យាយាមចាត់ថ្នាក់រូបភាពខ្លះពីអ៊ីនធឺណិត!


In [ ]:
# Sample images to classify
# These are from Unsplash (free stock photos)
test_images = [
    {
        "url": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400",
        "description": "A cat"
    },
    {
        "url": "https://images.unsplash.com/photo-1552053831-71594a27632d?w=400",
        "description": "A dog"
    },
    {
        "url": "https://images.unsplash.com/photo-1511919884226-fd3cad34687c?w=400",
        "description": "A car"
    },
]

print(f"🧪 Testing on {len(test_images)} images...")
print("=" * 70)

### ចំណាត់ថ្នាក់រាល់រូបភាព


In [ ]:
for i, img_data in enumerate(test_images, 1):
    print(f"\n📸 Image {i}: {img_data['description']}")
    print("-" * 70)
    
    try:
        # Load image
        img = load_image_from_url(img_data['url'])
        
        # Display image
        display(img.resize((200, 200)))  # Show smaller version
        
        # Classify
        results = classify_image(img)
        
        # Show predictions
        print("\n🎯 Top 5 Predictions:")
        for rank, (label, confidence) in enumerate(results, 1):
            # Create a visual bar
            bar_length = int(confidence * 50)
            bar = "█" * bar_length
            
            print(f"  {rank}. {label:20s} {confidence*100:5.2f}% {bar}")
        
    except Exception as e:
        print(f"❌ Error: {e}")

print("\n" + "=" * 70)

## ជំហានទី 5: ព្យាយាមរូបភាពផ្ទាល់ខ្លួនរបស់អ្នក!

ជំនួស URL ខាងក្រោមជាមួយ URL រូបភាពណាមួយដែលអ្នកចង់ចាត់ថ្នាក់។


In [ ]:
# Try your own image!
# Replace this URL with any image URL
custom_image_url = "https://images.unsplash.com/photo-1472491235688-bdc81a63246e?w=400"  # A flower

print("🖼️  Classifying your custom image...")
print("=" * 70)

try:
    # Load and show image
    img = load_image_from_url(custom_image_url)
    display(img.resize((300, 300)))
    
    # Classify
    results = classify_image(img)
    
    # Show results
    print("\n🎯 Top 5 Predictions:")
    print("-" * 70)
    for rank, (label, confidence) in enumerate(results, 1):
        bar_length = int(confidence * 50)
        bar = "█" * bar_length
        print(f"  {rank}. {label:20s} {confidence*100:5.2f}% {bar}")
    
    # Highlight top prediction
    top_label, top_confidence = results[0]
    print("\n" + "=" * 70)
    print(f"\n🏆 Best guess: {top_label} ({top_confidence*100:.2f}% confident)")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("   Make sure the URL points to a valid image!")

## 💡 តើមានអ្វីកើតឡើង?

1. **យើងបានបញ្ចូលម៉ូដែលដែលបានបណ្តុះមុន** - MobileNetV2 បានបណ្តុះលើរូបភាពលានលាន
2. **យើងបានកំណត់រូបភាពជាមុន** - បម្លែងទំហំ និងទ្រង់ទ្រាយសម្រាប់ម៉ូដែល
3. **ម៉ូដែលបានទស្សន៍ទាយ** - វាបានផ្តល់ព្រាប្បុលភាពសម្រាប់ ១០០០ ចំណាត់ថ្នាក់វត្ថុ
4. **យើងបានបកស្រាយលទ្ធផល** - បម្លែងលេខទៅជាអត្ថបទដែលមនុស្សអាចអានបាន

### ការយល់ដឹងអំពីពិន្ទុទំនុកចិត្ត

- **៩០-១០០%**: មានទំនុកចិត្តខ្លាំង (ភាគច្រើនត្រឹមត្រូវ)
- **៧០-៩០%**: មានទំនុកចិត្ត (ប្រហែលត្រឹមត្រូវ)
- **៥០-៧០%**: មានទំនុកចិត្តខ្លះៗ (ប្រហែលជាត្រឹមត្រូវ)
- **ក្រោម ៥០%**: មិនមានទំនុកចិត្តខ្លាំង (មិនប្រាកដ)

### ហេតុអ្វីបានជា ការទស្សន៍ទាយអាចខុស?

- **មុំឬពន្លឺមិនធម្មតា** - ម៉ូដែលបានបណ្តុះដោយរូបថតទំរង់ទូទៅ
- **វត្ថុច្រើនកន្លែង** - ម៉ូដែលនឹកស្រមៃមានវត្ថុមួយដើម
- **វត្ថុសម្បូរទ្រង់ទ្រាយកោសិកា** - ម៉ូដែលស្គាល់តែ ១០០០ ប្រភេទ
- **រូបភាពគុណភាពទាប** - រូបភាពមិនច្បាស់ឬម៉ូសេតារីមានការលំបាក


## 🚀 ជំហានបន្ទាប់

1. **សាកល្បងរូបភាពផ្សេងៗ:**
   - ស្វែងរករូបភាពនៅលើ [Unsplash](https://unsplash.com)
   - ចុចមែកខាងស្តាំ → "ចម្លងអាសយដ្ឋានរូបភាព" ដើម្បីទទួលបាន URL

2. **សាកល្បង:**
   - តើមានអ្វីកើតឡើងជាមួយសិល្បៈ abstract?
   - តើវាអាចរកឃើញវត្ថុពីមุมផ្សេងៗទេ?
   - តើវាដំណើរការយ៉ាងដូចម្តេចជាមួយវត្ថុច្រើន?

3. **រៀនបន្ថែម:**
   - ស្វែងយល់អំពី [មេរៀនទស្សនវិទ្យាកុំព្យូទ័រ](../lessons/4-ComputerVision/README.md)
   - រៀនបណ្ដុះបណ្ដាលម៉ាស៊ីនចាត់ថ្នាក់រូបភាពឯង
   - យល់ដឹងពីរបៀប CNNs (បណ្តាញប្រសាទបូកបញ្ចូល) ធ្វើការ

---

## 🎉 សូមអបអរសាទរ!

អ្នកទើបបង្កើតម៉ាស៊ីនចាត់ថ្នាក់រូបភាពដោយប្រើបណ្តាញប្រសាទទំនើប!

បច្ចេកវិទ្យានេះត្រូវបានប្រើសម្រាប់:
- Google Photos (ចាត់តាំងរូបថតរបស់អ្នក)
- ឡានបើកដោយខ្លួនឯង (ស្គាល់វត្ថុ)
- ការធ្វើវិភាគវេជ្ជសាស្រ្ត (វិភាគរូបថត X-ray)
- ការត្រួតពិនិត្យគុណភាព (រកកោសិលិ្អ)

បន្តស្វែងយល់ និងរៀនបន្ថែម! 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**កំណត់សម្គាល់**៖  
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខំប្រឹងប្រែងរកភាពត្រឹមត្រូវ សូមកត់សម្គាល់ថាការបកប្រែដោយស្វ័យប្រវត្តិសមត្ថភាពអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាតំណើរការរបស់វាគួរត្រូវបានគេចាត់ទុកថាជា ប្រភពផ្លូវការ។ សម្រាប់ព័ត៌មានសំខាន់ៗ គេបนะนำឲ្យប្រើការបកប្រែដោយមនុស្សឯកទេសជំនាញ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសផ្សេង ដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
